# Part 1 - Building Your First AI Agent with LangChain

*Last updated: November 2026*

In this lesson you will learn the LangChain building blocks of an agent:
1. **Prompt templates** - reusable prompts with variables.
2. **Chat models** - sending a message to an LLM.
3. **LLM chains** - combining a prompt template with a model.
4. **Tools and agents** - a ReAct agent that decides when to search the web or call your own tool.
5. **Tools built from chains** - a summarizer tool used by an agent.

## How to run this notebook

1. Set up the course environment once, in the course folder: `uv sync`.
   New to coding? Follow `BEGINNER_SETUP.md` step by step. Otherwise see `LOCAL_SETUP.MD`.
2. Select the **`.venv`** Python kernel (*Python 3.12*) for this notebook.
3. Copy `.env.example` (course folder) to `.env` and add your keys. The notebook finds `.env` automatically.

**Keys and settings used in this notebook**
- `OPENAI_API_KEY` (required)
- `TAVILY_API_KEY` (optional)

Run the cells from top to bottom.

> **Compatibility update (differs slightly from the recording)**
> - The recorded `gpt-3.5-turbo` models are legacy models. This notebook uses **`gpt-4o-mini`**, which is cheaper and current.
> - `ChatOpenAI` is now imported from `langchain_openai` (the old `langchain.chat_models` import is deprecated).
> - Chains and agents are called with `.invoke(...)` instead of the deprecated `.run(...)`.
> - DuckDuckGo search still needs no API key. If it is rate-limited, add an optional `TAVILY_API_KEY` and the notebook switches to Tavily search automatically.
> - The search tool is wrapped in a small `search_with_fallback` function. If a search fails, the agent receives an error message
>   and can try again, instead of the whole run stopping with an exception.
> - The last cell prints package versions with Python's `importlib.metadata` instead of the `pip show` command used in the video,
>   because the course environment is managed by uv and does not include pip.

In [ ]:
# Setup check: run this cell first.
# Nothing is installed here. All packages were installed once with `uv sync` (see LOCAL_SETUP.MD).
import sys
from pathlib import Path

python_version = sys.version.split()[0]

if sys.version_info[:2] != (3, 12):
    print(f"Warning: this course is tested on Python 3.12, but this kernel runs Python {python_version}.")
    print("Select the .venv kernel - see 'How to run this notebook' above.")
elif Path(sys.prefix).name != ".venv":
    print("Warning: this kernel is not the course .venv environment, so some packages may be missing.")
    print("Select the .venv kernel - see 'How to run this notebook' above.")
else:
    print("Ready: course .venv kernel | Python", python_version)

In [ ]:
# Load API keys from the .env file, so keys are never written in the notebook.
# load_dotenv() looks for .env in this folder and then in each parent folder (the course folder).
import os

from dotenv import load_dotenv

load_dotenv()

REQUIRED_KEYS = ['OPENAI_API_KEY']

missing = [key for key in REQUIRED_KEYS if not os.getenv(key)]
if missing:
    print("Missing required keys:", ", ".join(missing), "- see 'How to run this notebook' above.")
else:
    print("All required API keys are loaded.")

In [ ]:
# LangChain
from langchain.tools import Tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chains import LLMChain
from langchain.agents import initialize_agent, AgentType

In [ ]:
# Hide library deprecation warnings so the lesson output stays readable
import warnings
warnings.filterwarnings("ignore")

## 1. Prompt templates

In [ ]:
# Define a simple prompt template
prompt_template = PromptTemplate(
    input_variables=["name"],
    template="Hello, {name}! How can I help you today?"
)

# Generate the prompt by filling in the variable
formatted_prompt = prompt_template.format(name="David")
print(formatted_prompt)  # Output: Hello, David! How can I help you today?

## 2. Working with a chat model

In [ ]:
# Initialize the OpenAI chat model
chat_model = ChatOpenAI(model="gpt-4o-mini")

# Send a message to the model and get the response
response = chat_model.invoke("What is Capital of USA?")
print(response.content)  # Output: The capital of the United States of America is Washington, D.C.

## 3. LLM chains

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini")

learn_template = """
I want you to act as a consultant for a AI training
Return a list of topics and why it is important to learn in given area of AI
The description should be relevant to recent advancement in AI
What are some good topics to learn in {AI_topic}
"""

prompt_template = PromptTemplate(
    input_variables=["AI_topic"],
    template=learn_template,
)

description = "Please suggest me topics to learn in Deep learning"

chain = LLMChain(llm=llm, prompt=prompt_template)

result = chain.invoke({"AI_topic": description})
print(result["text"])

## 4. Tools and a ReAct agent

The agent receives a list of tools and decides, step by step, which tool to use (*Thought → Action → Observation*).

In [ ]:
import os

# Web search tool - DuckDuckGo needs no API key
web_search = DuckDuckGoSearchRun()

# Optional: DuckDuckGo sometimes rate-limits searches, especially on shared networks.
# If you added a TAVILY_API_KEY, Tavily search is used instead.
if os.getenv("TAVILY_API_KEY"):
    from langchain_tavily import TavilySearch
    web_search = TavilySearch(max_results=2)


# Fallback: if a search fails, return a message to the agent instead of stopping the run.
# The agent reads it as an observation and can try another query or answer without searching.
def search_with_fallback(query: str) -> str:
    try:
        return str(web_search.invoke(query))
    except Exception as error:
        return f"Search failed ({type(error).__name__}). Try a different query, or answer without searching."


search_tool = Tool.from_function(
    func=search_with_fallback,
    name=web_search.name,
    description=web_search.description,
)
print("Search tool:", search_tool.name)

In [ ]:
# Define a simple tool
def my_tool_function(query: str) -> str:
    return f"Tool response: {query}"

# Create a tool from the function
my_tool = Tool.from_function(func=my_tool_function, name="simple_tool", description="A simple tool")

# Initialize the LLM and the agent
llm = ChatOpenAI(model="gpt-4o-mini")
tools = [search_tool, my_tool]

# Create an agent
agent = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)

# Use the agent - it chooses whether to use a tool or answer directly
response = agent.invoke({"input": "What's the weather like today in London?"})
print(response["output"])

## 5. A summarizer tool built from an LLM chain

In [ ]:
prompt_template = "Summarize the following content: {content}"
llm = ChatOpenAI(model="gpt-4o-mini")

llm_chain = LLMChain(
    llm=llm,
    prompt=PromptTemplate.from_template(prompt_template)
)


def summarize(content: str) -> str:
    return llm_chain.invoke({"content": content})["text"]


summarize_tool = Tool.from_function(
    func=summarize,
    name="Summarizer",
    description="Summarizes a web page"
)

In [ ]:
tools = [search_tool, summarize_tool]

agent = initialize_agent(
    tools=tools,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    llm=llm,
    verbose=True,
    handle_parsing_errors=True
)

In [ ]:
prompt = """Please tell me how to become a good footballer"""
result = agent.invoke({"input": prompt})
print(result["output"])

## Check the installed package versions

In [ ]:
# Show the versions of the main packages used in this lesson
# (uv environments do not include pip, so we read the versions from Python instead of `pip show`)
from importlib.metadata import version

for package in ["langchain", "langchain-openai", "openai"]:
    print(f"{package}: {version(package)}")